In [2]:
import sys
sys.path.append("/Users/emilieyu/endotehelial-masboss")

import pandas as pd
from scipy.interpolate import NearestNDInterpolator


from src.config import load_abm_sim_cfg
from src.paths import BM_RESULTS_DIR
sim_cfg = load_abm_sim_cfg()

from abm.rho_lookup_table import RhoLookupTable

## LUT Diagnostic Tests

Changed LUT to be built on input recruitment probabilities rather than output DSP/TJP1/JCAD probabilities. delta_RhoC_max should be changed?

In [2]:
path = BM_RESULTS_DIR / sim_cfg['files']['recruitment_csv']
df = pd.read_csv(path).dropna(axis=1).round(3)

In [10]:
""" Check Rho against interpolators. """
def _build(df):
    df_filtered = df[['RhoA', 'RhoC',
                        'p1_name','p1_value', # DSP
                        'p2_name','p2_value', # TJP1
                        'p3_name','p3_value']] # JCAD
    
    long_df = pd.DataFrame({
        'RhoA': pd.concat([df_filtered['RhoA']]*3, ignore_index=True),
        'RhoC': pd.concat([df_filtered['RhoC']]*3, ignore_index=True),
        'name': pd.concat([df_filtered['p1_name'], df_filtered['p2_name'], df_filtered['p3_name']], ignore_index=True),
        'value': pd.concat([df_filtered['p1_value'], df_filtered['p2_value'], df_filtered['p3_value']], ignore_index=True),
    })

    recr_df = long_df.pivot_table(index=['RhoA','RhoC'],
                        columns='name',
                        values='value',
                        aggfunc='first').reset_index()

    points = recr_df[['$DSP_recruitment','$TJP1_recruitment','$JCAD_recruitment']].values

    rhoA_interp = NearestNDInterpolator(points, recr_df['RhoA'].values)
    rhoC_interp = NearestNDInterpolator(points, recr_df['RhoC'].values)

    print(f">>> DEBUG: Successfully built interpolators")

    return rhoA_interp, rhoC_interp

print(df[['p1_name','p1_value','p2_name','p2_value','p3_name','p3_value','DSP','TJP1','JCAD','RhoA','RhoC']].head(7))
mask = (df['p1_value']==0) & (df['p2_value']==0) & (df['p3_value']==0)
print("\nRow with input (0,0,0):")
print(df[mask][['p1_value','p2_value','p3_value','RhoA','RhoC']])

rhoA_interp, rhoC_interp = _build(df)

print(f"\nInterpolator at (0,0,0): RhoA={float(rhoA_interp([0,0,0]))}, RhoC={float(rhoC_interp([0,0,0]))}")
print(f"Interpolator at (0,0,0.2): RhoA={float(rhoA_interp([0,0,0.2]))}, RhoC={float(rhoC_interp([0,0,0.2]))}")



            p1_name  p1_value            p2_name  p2_value            p3_name  \
0  $DSP_recruitment       0.0  $TJP1_recruitment       0.0  $JCAD_recruitment   
1  $DSP_recruitment       0.0  $TJP1_recruitment       0.0  $JCAD_recruitment   
2  $DSP_recruitment       0.0  $TJP1_recruitment       0.0  $JCAD_recruitment   
3  $DSP_recruitment       0.0  $TJP1_recruitment       0.0  $JCAD_recruitment   
4  $DSP_recruitment       0.0  $TJP1_recruitment       0.0  $JCAD_recruitment   
5  $DSP_recruitment       0.0  $TJP1_recruitment       0.0  $JCAD_recruitment   
6  $DSP_recruitment       0.0  $TJP1_recruitment       0.2  $JCAD_recruitment   

   p3_value    DSP   TJP1   JCAD   RhoA   RhoC  
0       0.0  0.004  0.003  0.003  0.463  0.437  
1       0.2  0.003  0.003  0.279  0.476  0.443  
2       0.4  0.003  0.004  0.451  0.450  0.447  
3       0.6  0.003  0.005  0.558  0.460  0.451  
4       0.8  0.004  0.002  0.630  0.441  0.461  
5       1.0  0.002  0.003  0.669  0.463  0.453  
6       

/var/folders/6g/q6rvxn7j5zz8mmsx5fy77vj80000gn/T/ipykernel_230/1619154866.py:36: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print(f"\nInterpolator at (0,0,0): RhoA={float(rhoA_interp([0,0,0]))}, RhoC={float(rhoC_interp([0,0,0]))}")
/var/folders/6g/q6rvxn7j5zz8mmsx5fy77vj80000gn/T/ipykernel_230/1619154866.py:37: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print(f"Interpolator at (0,0,0.2): RhoA={float(rhoA_interp([0,0,0.2]))}, RhoC={float(rhoC_interp([0,0,0.2]))}")


In [12]:
""" Checking ranges + max rhoa/rhoc """

print("RhoA range:", df['RhoA'].min(), "to", df['RhoA'].max())
print("RhoC range:", df['RhoC'].min(), "to", df['RhoC'].max())

print(f"\nrhoa_rest = 0.463")
print(f"rhoc_rest = 0.437")
print(f"delta_RhoA_max = {df['RhoA'].max() - 0.463:.3f}")
print(f"delta_RhoC_max = {df['RhoC'].max() - 0.437:.3f}")

# But we only query up to p_max=0.67, so check the REACHABLE range
mask = (df['p1_value'] <= 0.8) & (df['p2_value'] <= 0.8) & (df['p3_value'] <= 0.8)
reachable = df[mask]
print(f"\nReachable range (inputs <= 0.8, nearest to Hill max 0.67):")
print(f"  RhoA: {reachable['RhoA'].min():.3f} to {reachable['RhoA'].max():.3f}")
print(f"  RhoC: {reachable['RhoC'].min():.3f} to {reachable['RhoC'].max():.3f}")
print(f"  delta_RhoA_max (reachable) = {reachable['RhoA'].max() - 0.463:.3f}")
print(f"  delta_RhoC_max (reachable) = {reachable['RhoC'].max() - 0.437:.3f}")

# What row gives max RhoC?
max_rhoc_row = df.loc[df['RhoC'].idxmax()]
print(f"\nMax RhoC row:")
print(f"  inputs: DSP={max_rhoc_row['p1_value']}, TJP1={max_rhoc_row['p2_value']}, JCAD={max_rhoc_row['p3_value']}")
print(f"  RhoA={max_rhoc_row['RhoA']}, RhoC={max_rhoc_row['RhoC']}")

# What row gives max RhoA?
max_rhoa_row = df.loc[df['RhoA'].idxmax()]
print(f"\nMax RhoA row:")
print(f"  inputs: DSP={max_rhoa_row['p1_value']}, TJP1={max_rhoa_row['p2_value']}, JCAD={max_rhoa_row['p3_value']}")
print(f"  RhoA={max_rhoa_row['RhoA']}, RhoC={max_rhoa_row['RhoC']}")

RhoA range: 0.259 to 0.801
RhoC range: 0.263 to 0.802

rhoa_rest = 0.463
rhoc_rest = 0.437
delta_RhoA_max = 0.338
delta_RhoC_max = 0.365

Reachable range (inputs <= 0.8, nearest to Hill max 0.67):
  RhoA: 0.270 to 0.765
  RhoC: 0.287 to 0.777
  delta_RhoA_max (reachable) = 0.302
  delta_RhoC_max (reachable) = 0.340

Max RhoC row:
  inputs: DSP=0.0, TJP1=1.0, JCAD=1.0
  RhoA=0.261, RhoC=0.802

Max RhoA row:
  inputs: DSP=1.0, TJP1=0.0, JCAD=1.0
  RhoA=0.801, RhoC=0.263


In [3]:
lut = RhoLookupTable(sim_cfg, BM_RESULTS_DIR)

>>> DEBUG: Successfully loaded recruitment parameter sweep data.
>>> DEBUG: Successfully built interpolators
LUT ready | rest: RhoA=0.463 RhoC=0.437


In [10]:
lut.query(0.4, 0.4, 0.1)

(0.511, 0.5195000000000001)